# LINet3 Transfer Learning: ScanNet Pretrain → SUN RGB-D Fine-tune

**Two-phase training pipeline:**
1. Pretrain on ScanNet 20-category (with official train/val split)
2. Transfer backbone to SUN RGB-D 19-category (train + test)

---

## Checklist Before Running:

- [ ] **Enable A100 GPU:** Runtime > Change runtime type > A100
- [ ] **Upload ScanNet dataset to Drive:** `MyDrive/datasets/scannet_pretrain_256.tar.gz`
- [ ] **Upload SUN dataset to Drive:** `MyDrive/datasets/sunrgbd_19_traintest.tar.gz`

## 1. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability and specs
import torch
import subprocess

print("=" * 60)
print("GPU VERIFICATION")
print("=" * 60)

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU Device: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")

    gpu_name = torch.cuda.get_device_name(0)
    if 'A100' in gpu_name:
        print("\nA100 GPU detected - optimal for training")
    elif 'V100' in gpu_name:
        print("\nV100 GPU detected - good for training (slower than A100)")
    elif 'T4' in gpu_name:
        print("\nT4 GPU detected - will be slower, consider upgrading to A100")
    else:
        print(f"\nGPU: {gpu_name}")
else:
    print("\nNO GPU DETECTED!")
    print("Enable GPU: Runtime -> Change runtime type -> Hardware accelerator: GPU")
    raise RuntimeError("GPU is required for training")

print("\n" + "=" * 60)

In [ ]:
!nvidia-smi

## 2. Mount Google Drive

In [ ]:
from google.colab import drive
import os
from pathlib import Path

# Mount Google Drive
drive.mount('/content/drive')

print("\nGoogle Drive mounted successfully!")
print(f"\nDrive contents:")
!ls -la /content/drive/MyDrive/ | head -20

## 3. Clone Repository to Local Disk (Fast I/O)

In [ ]:
import os
from pathlib import Path

# Configuration
PROJECT_NAME = "Multi-Stream-Neural-Networks"
GITHUB_REPO = "https://github.com/clingergab/Multi-Stream-Neural-Networks.git"
LOCAL_REPO_PATH = f"/content/{PROJECT_NAME}"

print("=" * 60)
print("REPOSITORY SETUP")
print("=" * 60)

os.chdir('/content')

if Path(LOCAL_REPO_PATH).exists() and Path(f"{LOCAL_REPO_PATH}/.git").exists():
    print(f"Repo already exists: {LOCAL_REPO_PATH}")
    os.chdir(LOCAL_REPO_PATH)
    !git pull
else:
    if Path(LOCAL_REPO_PATH).exists():
        !rm -rf {LOCAL_REPO_PATH}
    print(f"Cloning from {GITHUB_REPO}...")
    !git clone {GITHUB_REPO} {LOCAL_REPO_PATH}
    if not Path(LOCAL_REPO_PATH).exists():
        raise RuntimeError(f"Failed to clone repository")
    os.chdir(LOCAL_REPO_PATH)

print(f"\nWorking directory: {os.getcwd()}")
!ls -la {LOCAL_REPO_PATH}
print("\n" + "=" * 60)

## 4. Install Dependencies

In [ ]:
# Install required packages
print("Installing dependencies...")

!pip install -q h5py tqdm matplotlib seaborn ray[tune] kornia thop

# Verify installations
import h5py
import tqdm
import matplotlib
import seaborn
import kornia
import thop

print("All dependencies installed!")
print(f"   h5py: {h5py.__version__}")
print(f"   matplotlib: {matplotlib.__version__}")
print(f"   kornia: {kornia.__version__}")
print(f"   thop: {thop.__version__}")

## 5. Setup Python Path & Imports

In [ ]:
import sys
import os
import json
import copy
import math
import shutil
import warnings
import random
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime
from collections import Counter

# Remove cached modules
modules_to_reload = [k for k in sys.modules.keys() if k.startswith('src.')]
for module in modules_to_reload:
    del sys.modules[module]

# Add project to Python path
project_root = '/content/Multi-Stream-Neural-Networks'
if project_root not in sys.path:
    sys.path.insert(0, project_root)

# Model
from src.models.linear_integration.li_net3 import li_resnet18
from src.models.linear_integration.li_net3.conv import LIConv2d, LIBatchNorm2d
from src.models.linear_integration.li_net3.container import LIReLU
from src.models.linear_integration.li_net3.pooling import LIMaxPool2d, LIAdaptiveAvgPool2d
from src.models.common.model_helpers import load_pretrained_backbone, save_checkpoint

# Datasets
from src.data_utils.scannet_pretrain_dataset import (
    ScanNetPretrainDataset, _load_norm_stats as scannet_load_norm_stats,
    _load_class_names as scannet_load_class_names, _discover_samples,
)
from src.data_utils.sunrgbd_dataset import get_sunrgbd_dataloaders

# Training
from src.training.augmentation_config import AugmentationConfig
from src.training.optimizers import create_stream_optimizer
from src.training.schedulers import setup_scheduler
from src.utils.seed import set_seed

# Visualization
from src.utils.visualization import (
    FeatureMapVisualizer,
    StreamContributionVisualizer,
    StreamGradCAM,
    IntegrationWeightVisualizer,
    find_misclassified,
    compare_samples,
    StreamRedundancyAnalyzer,
    PerClassDominanceAnalyzer,
    ActivationDivergenceAnalyzer,
    IntegrationWeightEvolutionVisualizer,
    reset_bn_stats,
)

print("All imports successful!")

In [ ]:
SEED = 42
DETERMINISTIC = False
set_seed(SEED, deterministic=DETERMINISTIC)
print(f"Seed: {SEED}, Deterministic: {DETERMINISTIC}")

## 6. Copy ScanNet Dataset

Checks if `/dev/shm` has >40 GB free. If yes, extracts to RAM disk for fastest I/O. Otherwise extracts to local SSD (`/content/`).

In [ ]:
DRIVE_SCANNET_TAR = "/content/drive/MyDrive/datasets/scannet_pretrain_256.tar.gz"

print("=" * 60)
print("SCANNET 20-CATEGORY PRETRAIN DATASET SETUP")
print("=" * 60)

# Check /dev/shm free space
shm_stat = shutil.disk_usage("/dev/shm")
shm_free_gb = shm_stat.free / (1024**3)
print(f"/dev/shm free space: {shm_free_gb:.1f} GB")

if shm_free_gb > 40:
    SCANNET_DATA_PATH = "/dev/shm/scannet_pretrain_256"
    extract_target = "/dev/shm"
    print(f"Sufficient RAM \u2014 extracting to RAM disk: {SCANNET_DATA_PATH}")
else:
    SCANNET_DATA_PATH = "/content/scannet_pretrain_256"
    extract_target = "/content"
    print(f"Insufficient RAM ({shm_free_gb:.1f} GB < 40 GB) \u2014 extracting to local SSD: {SCANNET_DATA_PATH}")

if Path(SCANNET_DATA_PATH).exists():
    print(f"\nAlready on local disk: {SCANNET_DATA_PATH}")
    train_dir = Path(f"{SCANNET_DATA_PATH}/train")
    if train_dir.exists():
        train_count = sum(1 for _ in train_dir.rglob("*_rgb.pt"))
        print(f"  Train samples: {train_count}")
elif Path(DRIVE_SCANNET_TAR).exists():
    print(f"\nFound on Drive: {DRIVE_SCANNET_TAR}")
    tar_name = Path(DRIVE_SCANNET_TAR).name
    local_tar = f"{extract_target}/{tar_name}"
    !rsync -ah --info=progress2 {DRIVE_SCANNET_TAR} {local_tar}
    print(f"\nExtracting...")
    !tar -xzf {local_tar} -C {extract_target}/ 2>&1 | grep -v "Ignoring unknown extended header"
    !rm {local_tar}
    train_dir = Path(f"{SCANNET_DATA_PATH}/train")
    train_count = sum(1 for _ in train_dir.rglob("*_rgb.pt"))
    print(f"Extracted. Train samples: {train_count}")
else:
    raise FileNotFoundError(f"Dataset not found at {DRIVE_SCANNET_TAR}")

print(f"\nScanNet dataset ready at: {SCANNET_DATA_PATH}")

---
# Phase 1: ScanNet Pretraining

---

## 7. ScanNet Configuration

Fill in hyperparameters from HPO results before running.

In [ ]:
STREAM_LABELS = {0: 'RGB', 1: 'Depth'}

# ======================== SCANNET DATASET ========================
SCANNET_DATASET_CONFIG = {
    'data_root': SCANNET_DATA_PATH,
    'batch_size': 128,
    'num_workers': 4,
    'seed': SEED,
}

SCANNET_AUGMENTATION_CONFIG = AugmentationConfig(
    rgb_aug_prob=0.5,      # TODO: fill from HPO
    rgb_aug_mag=0.5,       # TODO: fill from HPO
    depth_aug_prob=0.5,    # TODO: fill from HPO
    depth_aug_mag=0.5,     # TODO: fill from HPO
)

# ======================== SCANNET MODEL ========================
SCANNET_MODEL_CONFIG = {
    'num_classes': 20,
    'stream_input_channels': [3, 1],
    'width_multiplier': 0.75,
    'dropout_p': 0.25,     # TODO: fill from HPO
    'device': 'cuda',
    'use_amp': True,
}

# ======================== SCANNET OPTIMIZER ========================
SCANNET_OPTIMIZER_CONFIG = {
    'stream_lrs': [1e-3, 1e-3],           # TODO: fill from HPO
    'stream_weight_decays': [5e-5, 5e-5],  # TODO: fill from HPO
    'shared_lr': 1e-3,                     # TODO: fill from HPO
    'integration_weight_decay': 5e-5,      # TODO: fill from HPO
}

SCANNET_SCHEDULER_CONFIG = {
    'scheduler_type': 'cosine',
    't_max': 110,
    'eta_min': [1e-6, 1e-6, 1e-6, 1e-6],  # TODO: fill from HPO
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2,
}

# ======================== SCANNET TRAINING ========================
SCANNET_TRAIN_CONFIG = {
    'epochs': 115,
    'grad_clip_norm': 1.5,             # TODO: fill from HPO
    'early_stopping': True,
    'patience': 15,
    'monitor': 'val_mca',
    'label_smoothing': 0.05,           # TODO: fill from HPO
    'modality_dropout': True,
    'modality_dropout_start': 0,
    'modality_dropout_ramp': 20,
    'modality_dropout_rate': 0.05,
    'stream_monitoring': True,
    'gradient_monitoring': True,
    'gradient_log_freq': 0,
    'track_integration_weights': True,
    'integration_snapshot_freq': 10,
}

# Print summary
print("ScanNet Configuration:")
print(f"  Classes: {SCANNET_MODEL_CONFIG['num_classes']}")
print(f"  Epochs: {SCANNET_TRAIN_CONFIG['epochs']}")
print(f"  Batch size: {SCANNET_DATASET_CONFIG['batch_size']}")
print(f"  Width multiplier: {SCANNET_MODEL_CONFIG['width_multiplier']}")

## 8. ScanNet Dataset Loading

In [ ]:
print("=" * 60)
print("LOADING SCANNET DATASET")
print("=" * 60)

data_root = SCANNET_DATASET_CONFIG['data_root']

# Load dataset metadata
scannet_norm_stats = scannet_load_norm_stats(data_root)
scannet_class_names = scannet_load_class_names(data_root)
scannet_num_classes = len(scannet_class_names)

assert scannet_num_classes == SCANNET_MODEL_CONFIG['num_classes'], \
    f"Expected {SCANNET_MODEL_CONFIG['num_classes']} classes, found {scannet_num_classes}"

# Discover samples
train_samples = _discover_samples(os.path.join(data_root, 'train'), scannet_class_names)
val_samples = _discover_samples(os.path.join(data_root, 'val'), scannet_class_names)

print(f"Classes: {scannet_num_classes} ({scannet_class_names[:3]}...)")
print(f"Train samples: {len(train_samples)}")
print(f"Val samples:   {len(val_samples)}")

# Build datasets
g = torch.Generator().manual_seed(SEED)

train_dataset = ScanNetPretrainDataset(
    data_root=data_root,
    split='train',
    samples=train_samples,
    class_names=scannet_class_names,
    norm_stats=scannet_norm_stats,
    normalize=False,  # GPU will normalize after augmentation
    **SCANNET_AUGMENTATION_CONFIG.to_dict(),
)

val_dataset = ScanNetPretrainDataset(
    data_root=data_root,
    split='val',
    samples=val_samples,
    class_names=scannet_class_names,
    norm_stats=scannet_norm_stats,
    normalize=False,
)

# Stratified sampling for training (class imbalance)
all_labels = train_dataset.labels
label_counts = Counter(all_labels)
n_samples = len(all_labels)
class_weights_map = {label: n_samples / count for label, count in label_counts.items()}
sample_weights = torch.tensor(
    [class_weights_map[label] for label in all_labels], dtype=torch.float32
)

train_sampler = torch.utils.data.WeightedRandomSampler(
    weights=sample_weights,
    num_samples=n_samples,
    replacement=True,
    generator=g,
)

def worker_init_fn(worker_id):
    worker_seed = SEED + worker_id
    np.random.seed(worker_seed)
    random.seed(worker_seed)

scannet_train_loader = torch.utils.data.DataLoader(
    train_dataset,
    batch_size=SCANNET_DATASET_CONFIG['batch_size'],
    shuffle=False,
    sampler=train_sampler,
    num_workers=SCANNET_DATASET_CONFIG['num_workers'],
    prefetch_factor=2,
    persistent_workers=True,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
)

scannet_val_loader = torch.utils.data.DataLoader(
    val_dataset,
    batch_size=SCANNET_DATASET_CONFIG['batch_size'],
    shuffle=False,
    num_workers=SCANNET_DATASET_CONFIG['num_workers'],
    prefetch_factor=2,
    persistent_workers=False,
    pin_memory=True,
    worker_init_fn=worker_init_fn,
)

print(f"\nDataloaders created:")
print(f"  Train: {len(scannet_train_loader.dataset)} samples ({len(scannet_train_loader)} batches)")
print(f"  Val:   {len(scannet_val_loader.dataset)} samples ({len(scannet_val_loader)} batches)")
print("=" * 60)

## 9. Create ScanNet Model & Compile

In [ ]:
print("=" * 60)
print("SCANNET MODEL CREATION & COMPILATION")
print("=" * 60)

# Checkpoint directory on Drive (persistent)
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
scannet_checkpoint_dir = f"/content/drive/MyDrive/linet_checkpoints/scannet_pretrain_{timestamp}"
Path(scannet_checkpoint_dir).mkdir(parents=True, exist_ok=True)

SCANNET_TRAIN_CONFIG['save_path'] = f"{scannet_checkpoint_dir}/best_model.pt"
SCANNET_TRAIN_CONFIG['integration_snapshot_path'] = f"{scannet_checkpoint_dir}/integration_snapshots"
os.makedirs(SCANNET_TRAIN_CONFIG['integration_snapshot_path'], exist_ok=True)

# Create model
model = li_resnet18(
    num_classes=SCANNET_MODEL_CONFIG['num_classes'],
    stream_input_channels=SCANNET_MODEL_CONFIG['stream_input_channels'],
    width_multiplier=SCANNET_MODEL_CONFIG['width_multiplier'],
    dropout_p=SCANNET_MODEL_CONFIG['dropout_p'],
    device=SCANNET_MODEL_CONFIG['device'],
    use_amp=SCANNET_MODEL_CONFIG['use_amp'],
)

total_params = sum(p.numel() for p in model.parameters())
print(f"LINet3-ResNet18 created ({total_params:,} params)")

# Create optimizer
optimizer = create_stream_optimizer(
    model,
    optimizer_type='adamw',
    stream_lrs=SCANNET_OPTIMIZER_CONFIG['stream_lrs'],
    stream_weight_decays=SCANNET_OPTIMIZER_CONFIG['stream_weight_decays'],
    shared_lr=SCANNET_OPTIMIZER_CONFIG['shared_lr'],
    integration_weight_decay=SCANNET_OPTIMIZER_CONFIG['integration_weight_decay'],
)

# Create scheduler
scheduler = setup_scheduler(
    optimizer,
    scheduler_type=SCANNET_SCHEDULER_CONFIG['scheduler_type'],
    eta_min=SCANNET_SCHEDULER_CONFIG['eta_min'],
    t_max=SCANNET_SCHEDULER_CONFIG['t_max'],
    train_loader_len=len(scannet_train_loader),
    warmup_epochs=SCANNET_SCHEDULER_CONFIG['warmup_epochs'],
    warmup_start_factor=SCANNET_SCHEDULER_CONFIG['warmup_start_factor'],
)

# Compile
model.compile(
    optimizer=optimizer,
    scheduler=scheduler,
    loss='cross_entropy',
    label_smoothing=SCANNET_TRAIN_CONFIG['label_smoothing'],
    gpu_augmentation=True,
    norm_stats=scannet_norm_stats,
    **SCANNET_AUGMENTATION_CONFIG.to_dict(),
)

print("Model compiled!")
print(f"Checkpoint dir: {scannet_checkpoint_dir}")
print("=" * 60)

## 10. ScanNet Training

In [ ]:
warnings.filterwarnings(
    'ignore',
    message='The epoch parameter in `scheduler.step\\(\\)` was not necessary',
    category=UserWarning
)

print("=" * 60)
print("SCANNET PRETRAINING")
print("=" * 60)

scannet_history = model.fit(
    train_loader=scannet_train_loader,
    val_loader=scannet_val_loader,
    epochs=SCANNET_TRAIN_CONFIG['epochs'],
    verbose=True,
    save_path=SCANNET_TRAIN_CONFIG['save_path'],
    early_stopping=SCANNET_TRAIN_CONFIG['early_stopping'],
    patience=SCANNET_TRAIN_CONFIG['patience'],
    restore_best_weights=True,
    grad_clip_norm=SCANNET_TRAIN_CONFIG['grad_clip_norm'],
    stream_monitoring=SCANNET_TRAIN_CONFIG['stream_monitoring'],
    monitor=SCANNET_TRAIN_CONFIG['monitor'],
    modality_dropout=SCANNET_TRAIN_CONFIG['modality_dropout'],
    modality_dropout_start=SCANNET_TRAIN_CONFIG['modality_dropout_start'],
    modality_dropout_ramp=SCANNET_TRAIN_CONFIG['modality_dropout_ramp'],
    modality_dropout_rate=SCANNET_TRAIN_CONFIG['modality_dropout_rate'],
    gradient_monitoring=SCANNET_TRAIN_CONFIG['gradient_monitoring'],
    gradient_log_freq=SCANNET_TRAIN_CONFIG['gradient_log_freq'],
    track_integration_weights=SCANNET_TRAIN_CONFIG['track_integration_weights'],
    integration_snapshot_path=SCANNET_TRAIN_CONFIG['integration_snapshot_path'],
    integration_snapshot_freq=SCANNET_TRAIN_CONFIG['integration_snapshot_freq'],
)

print("\n" + "=" * 60)
print("SCANNET PRETRAINING COMPLETE!")
print("=" * 60)

In [ ]:
print("=" * 60)
print("SAVING SCANNET CHECKPOINT")
print("=" * 60)

# Save final model checkpoint to Drive
scannet_final_path = f"{scannet_checkpoint_dir}/final_model.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': model.optimizer.state_dict(),
    'scheduler_state_dict': model.scheduler.state_dict() if model.scheduler else None,
    'config': SCANNET_MODEL_CONFIG,
    'history': scannet_history,
}, scannet_final_path)

print(f"ScanNet model saved: {scannet_final_path}")

# Also save training history as JSON
scannet_history_path = f"{scannet_checkpoint_dir}/training_history.json"
json_history = {
    'train_loss': [float(x) for x in scannet_history['train_loss']],
    'train_accuracy': [float(x) for x in scannet_history['train_accuracy']],
    'train_mca': [float(x) for x in scannet_history.get('train_mca', [])],
    'val_loss': [float(x) for x in scannet_history.get('val_loss', [])],
    'val_accuracy': [float(x) for x in scannet_history.get('val_accuracy', [])],
    'val_mca': [float(x) for x in scannet_history.get('val_mca', [])],
    'learning_rates': [float(x) for x in scannet_history['learning_rates']],
    'model_config': SCANNET_MODEL_CONFIG,
    'dataset_config': {k: str(v) if not isinstance(v, (int, float, bool, type(None))) else v
                       for k, v in SCANNET_DATASET_CONFIG.items()},
    'optimizer_config': SCANNET_OPTIMIZER_CONFIG,
    'scheduler_config': SCANNET_SCHEDULER_CONFIG,
    'training_config': {k: str(v) if not isinstance(v, (int, float, bool, type(None))) else v
                        for k, v in SCANNET_TRAIN_CONFIG.items()},
    'augmentation_config': SCANNET_AUGMENTATION_CONFIG.to_dict(),
}
with open(scannet_history_path, 'w') as f:
    json.dump(json_history, f, indent=2)

print(f"Training history saved: {scannet_history_path}")
print("=" * 60)

## 11. ScanNet Training Curves

Training diagnostics: loss, accuracy, LR schedule, gradient norms, per-stream contribution, gradient health.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# Loss curve (train + val)
axes[0, 0].plot(scannet_history['train_loss'], label='Train Loss', linewidth=2)
if 'val_loss' in scannet_history and scannet_history['val_loss']:
    axes[0, 0].plot(scannet_history['val_loss'], label='Val Loss', linewidth=2, linestyle='--')
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# Accuracy curves (train + val + per-stream)
axes[0, 1].plot([acc*100 for acc in scannet_history['train_accuracy']], label='Train Acc', linewidth=2, color='green')
if 'val_accuracy' in scannet_history and scannet_history['val_accuracy']:
    axes[0, 1].plot([acc*100 for acc in scannet_history['val_accuracy']], label='Val Acc', linewidth=2, color='darkgreen', linestyle='--')
if 'train_mca' in scannet_history and scannet_history['train_mca']:
    axes[0, 1].plot([m*100 for m in scannet_history['train_mca']], label='Train MCA', linewidth=2, color='darkorange', linestyle=':')
if 'val_mca' in scannet_history and scannet_history['val_mca']:
    axes[0, 1].plot([m*100 for m in scannet_history['val_mca']], label='Val MCA', linewidth=2, color='red', linestyle=':')
stream_train_colors = ['skyblue', 'lightcoral', 'gold', 'lightgreen', 'plum']
for i in range(len(SCANNET_MODEL_CONFIG['stream_input_channels'])):
    color_idx = i % len(stream_train_colors)
    if f'stream_{i}_train_acc' in scannet_history:
        axes[0, 1].plot([acc*100 for acc in scannet_history[f'stream_{i}_train_acc']],
                    label=f'{STREAM_LABELS[i]} Train', linewidth=1, alpha=0.6, linestyle='--',
                    color=stream_train_colors[color_idx])
axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Accuracy (%)', fontsize=12)
axes[0, 1].set_yticks([20, 40, 60, 80, 100])
axes[0, 1].set_title('Accuracy', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=9, loc='lower right')
for y in range(0, 101, 10):
    axes[0, 1].axhline(y=y, color='gray', alpha=0.3, linewidth=0.5)
for y in range(5, 100, 10):
    axes[0, 1].axhline(y=y, color='gray', alpha=0.2, linewidth=0.5)
axes[0, 1].grid(True, axis='x', alpha=0.3)

# LR schedule
sampled_lrs = scannet_history['learning_rates'][::max(1, len(scannet_history['learning_rates'])//100)]
axes[0, 2].plot(sampled_lrs, linewidth=2, color='green', label='Base LR')
lr_colors = ['blue', 'red', 'orange', 'purple', 'brown']
for i in range(len(SCANNET_MODEL_CONFIG['stream_input_channels'])):
    color_idx = i % len(lr_colors)
    if f'stream_{i}_lr' in scannet_history:
        axes[0, 2].plot(scannet_history[f'stream_{i}_lr'], linewidth=1, alpha=0.7, linestyle='--',
                    color=lr_colors[color_idx], label=f'{STREAM_LABELS[i]} LR')
axes[0, 2].set_xlabel('Epoch', fontsize=12)
axes[0, 2].set_ylabel('Learning Rate', fontsize=12)
axes[0, 2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[0, 2].legend(fontsize=9, loc='upper right')
axes[0, 2].grid(True, alpha=0.3)

# Gradient norms
stream_val_colors = ['blue', 'red', 'orange', 'green', 'purple']
if 'gradient_norms' in scannet_history and scannet_history['gradient_norms']:
    grad_epochs = range(len(scannet_history['gradient_norms']))
    for i in range(len(SCANNET_MODEL_CONFIG['stream_input_channels'])):
        key = f'stream_{i}'
        norms = [d.get(key, {}).get('mean', 0) for d in scannet_history['gradient_norms']]
        color = stream_val_colors[i % len(stream_val_colors)]
        axes[1, 0].plot(grad_epochs, norms, label=f'{STREAM_LABELS[i]}', color=color, linewidth=1.5)
    shared_norms = [d.get('shared', {}).get('mean', 0) for d in scannet_history['gradient_norms']]
    axes[1, 0].plot(grad_epochs, shared_norms, label='Shared', color='gray', linewidth=1.5, linestyle='--')
    axes[1, 0].set_yscale('log')
    axes[1, 0].set_xlabel('Epoch', fontsize=12)
    axes[1, 0].set_ylabel('Gradient Norm (pre-clip, log)', fontsize=12)
    axes[1, 0].set_title('Per-Stream Gradient Norms (mean)', fontsize=14, fontweight='bold')
    axes[1, 0].legend(fontsize=9)
    axes[1, 0].grid(True, alpha=0.3)
else:
    axes[1, 0].text(0.5, 0.5, 'No gradient data', ha='center', va='center',
                    transform=axes[1, 0].transAxes, fontsize=12)
    axes[1, 0].set_title('Per-Stream Gradient Norms', fontsize=14, fontweight='bold')

# Per-stream contribution
contrib_keys = [f'stream_{i}_train_acc' for i in range(len(SCANNET_MODEL_CONFIG['stream_input_channels']))]
if contrib_keys[0] in scannet_history:
    n_streams = len(SCANNET_MODEL_CONFIG['stream_input_channels'])
    baseline_vals = scannet_history['train_accuracy']
    for i in range(n_streams):
        color = stream_val_colors[i % len(stream_val_colors)]
        other = (i + 1) % n_streams if n_streams == 2 else i
        other_vals = scannet_history[f'stream_{other}_train_acc']
        contrib = []
        epochs_eval = []
        for e, (other_acc, base) in enumerate(zip(other_vals, baseline_vals)):
            if not math.isnan(other_acc):
                contrib.append((base - other_acc) * 100)
                epochs_eval.append(e)
        axes[1, 1].plot(epochs_eval, contrib,
                       label=f'{STREAM_LABELS[i]}', color=color, linewidth=1.5, marker='o', markersize=3)
    axes[1, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[1, 1].set_xlabel('Epoch', fontsize=12)
    axes[1, 1].set_ylabel('Contribution (pp)', fontsize=12)
    axes[1, 1].set_title('Per-Stream Contribution\n(Baseline - Acc w/o Stream)', fontsize=14, fontweight='bold')
    axes[1, 1].legend(fontsize=9)
    axes[1, 1].grid(True, alpha=0.3)
else:
    axes[1, 1].text(0.5, 0.5, 'No stream data', ha='center', va='center',
                    transform=axes[1, 1].transAxes, fontsize=12)
    axes[1, 1].set_title('Per-Stream Contribution', fontsize=14, fontweight='bold')

# Gradient health
if 'gradient_health' in scannet_history and scannet_history['gradient_health']:
    axes[1, 2].axis('off')
    health_text = "Gradient Health Summary:\n\n"
    status_counts = {}
    for h in scannet_history['gradient_health']:
        status = h.get('status', 'unknown') if isinstance(h, dict) else str(h)
        status_counts[status] = status_counts.get(status, 0) + 1
    for status, count in sorted(status_counts.items(), key=lambda x: -x[1]):
        health_text += f"  {status}: {count} epochs\n"
    axes[1, 2].text(0.1, 0.9, health_text, transform=axes[1, 2].transAxes,
                    fontsize=10, verticalalignment='top', fontfamily='monospace')
    axes[1, 2].set_title('Gradient Health', fontsize=14, fontweight='bold')
else:
    axes[1, 2].text(0.5, 0.5, 'No gradient health data', ha='center', va='center',
                    transform=axes[1, 2].transAxes, fontsize=12)
    axes[1, 2].set_title('Gradient Health', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{scannet_checkpoint_dir}/training_diagnostics.pdf", dpi=150, bbox_inches='tight')
plt.show()

print(f"ScanNet training diagnostics saved to: {scannet_checkpoint_dir}/training_diagnostics.pdf")

## 12. ScanNet Integration Weight Evolution

In [ ]:
evo_viz = IntegrationWeightEvolutionVisualizer(stream_labels=STREAM_LABELS)

# Stream backbone weight norm evolution
if 'stream_weight_norms' in scannet_history:
    evo_viz.plot_stream_weight_norms(scannet_history, save_path=f"{scannet_checkpoint_dir}/stream_weight_evolution.pdf")
    print("Stream weight norm evolution saved.")
else:
    print("No stream weight norm data found.")

# Integration weight norm evolution
if 'integration_weight_norms' in scannet_history:
    evo_viz.plot_norm_evolution(scannet_history, save_path=f"{scannet_checkpoint_dir}/integration_weight_evolution.pdf")
    print("Integration weight norm evolution saved.")
else:
    print("No integration weight norm data found.")

# Full weight snapshots
snapshot_dir = SCANNET_TRAIN_CONFIG.get('integration_snapshot_path')
if snapshot_dir and os.path.isdir(snapshot_dir) and os.listdir(snapshot_dir):
    evo_viz.plot_snapshot_heatmaps(snapshot_dir, save_path=f"{scannet_checkpoint_dir}/integration_weight_snapshots.png")
    print("Integration weight snapshot heatmaps saved (full, PNG).")
    for layer_name in ['conv1', 'layer1']:
        evo_viz.plot_snapshot_heatmaps(
            snapshot_dir, layer_filter=layer_name,
            save_path=f"{scannet_checkpoint_dir}/integration_weight_snapshots_{layer_name}.pdf"
        )
    print("Integration weight snapshot heatmaps saved (conv1 + layer1, PDF).")
else:
    print("No integration weight snapshots found.")

---
# Phase 1 \u2192 Phase 2 Transition

Clean up ScanNet resources and prepare for SUN fine-tuning.

---

## 13. Cleanup & Transition

In [ ]:
print("=" * 60)
print("CLEANUP: TRANSITIONING TO SUN FINE-TUNING")
print("=" * 60)

# Store the checkpoint path for Phase 2
SCANNET_CHECKPOINT_PATH = scannet_final_path
print(f"ScanNet checkpoint: {SCANNET_CHECKPOINT_PATH}")

# Free GPU memory
del model, optimizer, scheduler
del scannet_train_loader, scannet_val_loader
del train_dataset, val_dataset
torch.cuda.empty_cache()

# Remove ScanNet data from disk/RAM
if os.path.exists(SCANNET_DATA_PATH):
    shutil.rmtree(SCANNET_DATA_PATH)
    print(f"Removed ScanNet data: {SCANNET_DATA_PATH}")

# Report freed memory
print(f"GPU memory allocated: {torch.cuda.memory_allocated() / 1024**2:.0f} MB")
print(f"GPU memory reserved:  {torch.cuda.memory_reserved() / 1024**2:.0f} MB")
print("\nReady for SUN RGB-D fine-tuning!")
print("=" * 60)

---
# Phase 2: SUN RGB-D Fine-tuning (with ScanNet Backbone)

---

## 14. Copy SUN RGB-D Dataset to RAM

SUN RGB-D is small (~2-3 GB) \u2014 always loads to `/dev/shm` (RAM disk).

In [ ]:
DRIVE_SUN_TAR = "/content/drive/MyDrive/datasets/sunrgbd_19_traintest.tar.gz"
SUN_DATA_PATH = "/dev/shm/sunrgbd_19_traintest"

print("=" * 60)
print("SUN RGB-D 19-CATEGORY DATASET SETUP")
print("=" * 60)

if Path(SUN_DATA_PATH).exists():
    print(f"Already on local disk: {SUN_DATA_PATH}")
elif Path(DRIVE_SUN_TAR).exists():
    print(f"Found on Drive: {DRIVE_SUN_TAR}")
    tar_name = Path(DRIVE_SUN_TAR).name
    local_tar = f"/dev/shm/{tar_name}"
    !rsync -ah --info=progress2 {DRIVE_SUN_TAR} {local_tar}
    print(f"\nExtracting...")
    !tar -xzf {local_tar} -C /dev/shm/ 2>&1 | grep -v "Ignoring unknown extended header"
    !rm {local_tar}
    print("Extracted.")
else:
    raise FileNotFoundError(f"Dataset not found at {DRIVE_SUN_TAR}")

# Verify
dataset_root = Path(SUN_DATA_PATH)
for split in ['train', 'test']:
    split_dir = dataset_root / split
    if split_dir.exists():
        rgb_count = len(list((split_dir / 'rgb').glob('*.png')))
        print(f"  {split}: {rgb_count} images")

print(f"\nSUN dataset ready at: {SUN_DATA_PATH}")
print("=" * 60)

## 15. SUN Configuration

Fill in hyperparameters from HPO results. **width_multiplier and stream_input_channels MUST match ScanNet config** for weight transfer.

In [ ]:
# ======================== SUN DATASET ========================
SUN_DATASET_CONFIG = {
    'data_root': SUN_DATA_PATH,
    'batch_size': 64,
    'num_workers': 5,
    'num_classes': 19,
    'seed': SEED,
}

SUN_AUGMENTATION_CONFIG = AugmentationConfig(
    rgb_aug_prob=1.0,      # TODO: fill from HPO
    rgb_aug_mag=1.0,       # TODO: fill from HPO
    depth_aug_prob=1.0,    # TODO: fill from HPO
    depth_aug_mag=1.0,     # TODO: fill from HPO
)

# ======================== SUN MODEL ========================
SUN_MODEL_CONFIG = {
    'num_classes': 19,
    'stream_input_channels': [3, 1],  # MUST match ScanNet
    'width_multiplier': 0.75,         # MUST match ScanNet
    'dropout_p': 0.5,     # TODO: fill from HPO
    'device': 'cuda',
    'use_amp': True,
}

# Validate architecture compatibility
assert SUN_MODEL_CONFIG['width_multiplier'] == SCANNET_MODEL_CONFIG['width_multiplier'], \
    "width_multiplier must match between ScanNet and SUN for weight transfer!"
assert SUN_MODEL_CONFIG['stream_input_channels'] == SCANNET_MODEL_CONFIG['stream_input_channels'], \
    "stream_input_channels must match between ScanNet and SUN for weight transfer!"

# ======================== SUN OPTIMIZER ========================
SUN_OPTIMIZER_CONFIG = {
    'stream_lrs': [7e-05, 1.5e-04],        # TODO: fill from HPO
    'shared_lr': 1.5e-04,                   # TODO: fill from HPO
    'stream_weight_decays': [6e-05, 4e-05], # TODO: fill from HPO
    'integration_weight_decay': 1e-04,      # TODO: fill from HPO
}

SUN_SCHEDULER_CONFIG = {
    'scheduler_type': 'cosine',
    't_max': 115,                            # TODO: fill from HPO
    's1_eta': 1e-06,                         # TODO: fill from HPO
    's2_eta': 2e-06,                         # TODO: fill from HPO
    'eta_min': 7e-07,                        # TODO: fill from HPO
    'warmup_epochs': 5,
    'warmup_start_factor': 0.2,
}

# ======================== SUN TRAINING ========================
SUN_TRAIN_CONFIG = {
    'epochs': 120,
    'grad_clip_norm': 0.8,             # TODO: fill from HPO
    'early_stopping': False,
    'restore_best_weights': True,
    'stream_monitoring': True,
    'modality_dropout': True,
    'modality_dropout_start': 5,
    'modality_dropout_ramp': 20,
    'modality_dropout_rate': 0.12,     # TODO: fill from HPO
    'label_smoothing': 0.12,           # TODO: fill from HPO
    'gradient_monitoring': True,
    'gradient_log_freq': 0,
    'track_integration_weights': True,
    'integration_snapshot_freq': 10,
    'monitor': 'val_mca',
}

# ======================== TRANSFER CONFIG ========================
TRANSFER_CONFIG = {
    'pretrained_checkpoint': SCANNET_CHECKPOINT_PATH,
    'freeze_backbone_epochs': 0,  # Set > 0 to freeze backbone for N warmup epochs
}

print("SUN Configuration:")
print(f"  Classes: {SUN_MODEL_CONFIG['num_classes']}")
print(f"  Epochs: {SUN_TRAIN_CONFIG['epochs']}")
print(f"  Batch size: {SUN_DATASET_CONFIG['batch_size']}")
print(f"  Freeze warmup epochs: {TRANSFER_CONFIG['freeze_backbone_epochs']}")
print(f"  Pretrained checkpoint: {TRANSFER_CONFIG['pretrained_checkpoint']}")

## 16. Load SUN Dataset

In [ ]:
# Verify dataset structure
print("=" * 60)
print("DATASET STRUCTURE VERIFICATION")
print("=" * 60)

dataset_root = Path(SUN_DATA_PATH)
for split in ['train', 'test']:
    split_dir = dataset_root / split
    if split_dir.exists():
        print(f"  {split}/")
        for modality in ['rgb', 'depth']:
            mod_dir = split_dir / modality
            if mod_dir.exists():
                print(f"    {modality}/ - {len(list(mod_dir.glob('*.png')))} images")

class_names_file = dataset_root / 'class_names.txt'
if class_names_file.exists():
    with open(class_names_file, 'r') as f:
        sun_class_names = [line.strip() for line in f]
    print(f"\nClasses ({len(sun_class_names)}):")
    for i, name in enumerate(sun_class_names):
        print(f"  {i}: {name}")

print("\n" + "=" * 60)
print("LOADING SUN RGB-D 19-CATEGORY DATASET")
print("=" * 60)

sun_train_loader, sun_val_loader, sun_test_loader = get_sunrgbd_dataloaders(
    data_root=SUN_DATASET_CONFIG['data_root'],
    batch_size=SUN_DATASET_CONFIG['batch_size'],
    num_workers=SUN_DATASET_CONFIG['num_workers'],
    seed=SUN_DATASET_CONFIG['seed'],
    **SUN_AUGMENTATION_CONFIG.to_dict(),
    stratified=True,
    normalize=True,
)

print(f"\nDataset loaded!")
print(f"  Train: {len(sun_train_loader.dataset)} samples ({len(sun_train_loader)} batches)")
print(f"  Test:  {len(sun_test_loader.dataset)} samples ({len(sun_test_loader)} batches)")
print(f"  Val:   {'None' if sun_val_loader is None else f'{len(sun_val_loader.dataset)} samples'}")

rgb_batch, depth_batch, label_batch = next(iter(sun_train_loader))
print(f"\nBatch shapes: RGB={rgb_batch.shape}, Depth={depth_batch.shape}, Labels={label_batch.shape}")
print("=" * 60)

## 17. Create SUN Model with Transfer Learning

In [ ]:
print("=" * 60)
print("SUN MODEL CREATION + BACKBONE TRANSFER")
print("=" * 60)

# Checkpoint directory on Drive
timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
sun_checkpoint_dir = f"/content/drive/MyDrive/linet_checkpoints/sun_transfer_{timestamp}"
Path(sun_checkpoint_dir).mkdir(parents=True, exist_ok=True)

SUN_TRAIN_CONFIG['save_path'] = f"{sun_checkpoint_dir}/best_model.pt"
SUN_TRAIN_CONFIG['integration_snapshot_path'] = f"{sun_checkpoint_dir}/integration_snapshots"
os.makedirs(SUN_TRAIN_CONFIG['integration_snapshot_path'], exist_ok=True)

# Create fresh model with SUN num_classes (19)
model = li_resnet18(
    num_classes=SUN_MODEL_CONFIG['num_classes'],
    stream_input_channels=SUN_MODEL_CONFIG['stream_input_channels'],
    width_multiplier=SUN_MODEL_CONFIG['width_multiplier'],
    dropout_p=SUN_MODEL_CONFIG['dropout_p'],
    device=SUN_MODEL_CONFIG['device'],
    use_amp=SUN_MODEL_CONFIG['use_amp'],
)

# Transfer pretrained backbone from ScanNet
print(f"\nLoading pretrained backbone from: {TRANSFER_CONFIG['pretrained_checkpoint']}")
transfer_info = load_pretrained_backbone(model, TRANSFER_CONFIG['pretrained_checkpoint'])

total_params = sum(p.numel() for p in model.parameters())
print(f"\nModel: LINet3-ResNet18 ({total_params:,} params)")
print(f"  Loaded: {len(transfer_info['loaded'])} keys (backbone)")
print(f"  Skipped: {len(transfer_info['skipped'])} keys (classifier head)")
print(f"Checkpoint dir: {sun_checkpoint_dir}")
print("=" * 60)

## 18. Compile SUN Model

In [ ]:
print("=" * 60)
print("SUN MODEL COMPILATION")
print("=" * 60)

# Create optimizer with SUN-specific config
optimizer = create_stream_optimizer(
    model,
    optimizer_type='adamw',
    stream_lrs=SUN_OPTIMIZER_CONFIG['stream_lrs'],
    stream_weight_decays=SUN_OPTIMIZER_CONFIG['stream_weight_decays'],
    shared_lr=SUN_OPTIMIZER_CONFIG['shared_lr'],
    integration_weight_decay=SUN_OPTIMIZER_CONFIG['integration_weight_decay'],
)

print(f"Optimizer: {optimizer.__class__.__name__}")
for i, group in enumerate(optimizer.param_groups):
    num_params = sum(p.numel() for p in group['params'])
    print(f"  Group {i+1}: lr={group['lr']:.2e}, wd={group['weight_decay']:.2e}, params={num_params:,}")

# Create scheduler
scheduler = setup_scheduler(
    optimizer,
    scheduler_type=SUN_SCHEDULER_CONFIG['scheduler_type'],
    epochs=SUN_SCHEDULER_CONFIG['t_max'],
    train_loader_len=len(sun_train_loader),
    t_max=SUN_SCHEDULER_CONFIG['t_max'],
    eta_min=[SUN_SCHEDULER_CONFIG['s1_eta'], SUN_SCHEDULER_CONFIG['s2_eta'],
             SUN_SCHEDULER_CONFIG['eta_min'], SUN_SCHEDULER_CONFIG['eta_min']],
    warmup_epochs=SUN_SCHEDULER_CONFIG['warmup_epochs'],
    warmup_start_factor=SUN_SCHEDULER_CONFIG['warmup_start_factor'],
)

# Compile with SUN normalization stats (NOT ScanNet stats!)
model.compile(
    optimizer=optimizer,
    scheduler=scheduler,
    loss='cross_entropy',
    label_smoothing=SUN_TRAIN_CONFIG['label_smoothing'],
    gpu_augmentation=False,
    **SUN_AUGMENTATION_CONFIG.to_dict(),
)

print("\nModel compiled with SUN-specific configuration!")
print("=" * 60)

## 19. Optional Backbone Freeze Warmup

If `freeze_backbone_epochs > 0`, freeze the pretrained backbone and train only the classifier head for a few epochs. This lets the new head calibrate before the backbone starts adapting.

In [ ]:
freeze_epochs = TRANSFER_CONFIG['freeze_backbone_epochs']

if freeze_epochs > 0:
    print("=" * 60)
    print(f"BACKBONE FREEZE WARMUP ({freeze_epochs} epochs)")
    print("=" * 60)

    # Freeze all parameters except the classifier head
    frozen_count = 0
    for name, param in model.named_parameters():
        if not name.startswith('fc.'):
            param.requires_grad = False
            frozen_count += 1

    trainable_count = sum(1 for p in model.parameters() if p.requires_grad)
    print(f"  Frozen: {frozen_count} parameters")
    print(f"  Trainable: {trainable_count} parameters (classifier head only)")

    # Train head only -- NO modality dropout (need stable features)
    warmup_history = model.fit(
        train_loader=sun_train_loader,
        val_loader=None,
        epochs=freeze_epochs,
        verbose=True,
        grad_clip_norm=SUN_TRAIN_CONFIG['grad_clip_norm'],
        stream_monitoring=False,
        modality_dropout=False,
        gradient_monitoring=False,
        track_integration_weights=False,
    )

    # Unfreeze all parameters
    for param in model.parameters():
        param.requires_grad = True

    unfrozen_count = sum(1 for p in model.parameters() if p.requires_grad)
    print(f"\n  All {unfrozen_count} parameters unfrozen.")

    # Re-create optimizer and scheduler for full training (new param groups)
    optimizer = create_stream_optimizer(
        model,
        optimizer_type='adamw',
        stream_lrs=SUN_OPTIMIZER_CONFIG['stream_lrs'],
        stream_weight_decays=SUN_OPTIMIZER_CONFIG['stream_weight_decays'],
        shared_lr=SUN_OPTIMIZER_CONFIG['shared_lr'],
        integration_weight_decay=SUN_OPTIMIZER_CONFIG['integration_weight_decay'],
    )

    remaining_epochs = SUN_TRAIN_CONFIG['epochs'] - freeze_epochs
    scheduler = setup_scheduler(
        optimizer,
        scheduler_type=SUN_SCHEDULER_CONFIG['scheduler_type'],
        epochs=remaining_epochs,
        train_loader_len=len(sun_train_loader),
        t_max=remaining_epochs,
        eta_min=[SUN_SCHEDULER_CONFIG['s1_eta'], SUN_SCHEDULER_CONFIG['s2_eta'],
                 SUN_SCHEDULER_CONFIG['eta_min'], SUN_SCHEDULER_CONFIG['eta_min']],
        warmup_epochs=SUN_SCHEDULER_CONFIG['warmup_epochs'],
        warmup_start_factor=SUN_SCHEDULER_CONFIG['warmup_start_factor'],
    )

    model.compile(
        optimizer=optimizer,
        scheduler=scheduler,
        loss='cross_entropy',
        label_smoothing=SUN_TRAIN_CONFIG['label_smoothing'],
        gpu_augmentation=False,
        **SUN_AUGMENTATION_CONFIG.to_dict(),
    )

    print(f"  Re-compiled for {remaining_epochs} remaining epochs.")
    print("=" * 60)
else:
    warmup_history = None
    remaining_epochs = SUN_TRAIN_CONFIG['epochs']
    print("Backbone freeze warmup: DISABLED (freeze_backbone_epochs = 0)")

## 20. SUN Full Training

In [ ]:
def merge_histories(h1, h2):
    """Merge two training history dicts (e.g. warmup + full training)."""
    if h1 is None:
        return h2
    if h2 is None:
        return h1
    merged = {}
    all_keys = set(h1.keys()) | set(h2.keys())
    for key in all_keys:
        v1 = h1.get(key)
        v2 = h2.get(key)
        if v1 is None and v2 is None:
            merged[key] = None
        elif v1 is None:
            merged[key] = v2
        elif v2 is None:
            merged[key] = v1
        elif isinstance(v1, dict) and isinstance(v2, dict):
            merged[key] = merge_histories(v1, v2)
        elif isinstance(v1, list) and isinstance(v2, list):
            merged[key] = v1 + v2
        else:
            merged[key] = v2
    return merged


warnings.filterwarnings(
    'ignore',
    message='The epoch parameter in `scheduler.step\\(\\)` was not necessary',
    category=UserWarning
)

print("=" * 60)
print("SUN RGB-D FINE-TUNING (WITH SCANNET BACKBONE)")
print("=" * 60)

sun_history = model.fit(
    train_loader=sun_train_loader,
    val_loader=None,
    epochs=remaining_epochs,
    verbose=True,
    save_path=SUN_TRAIN_CONFIG['save_path'],
    early_stopping=SUN_TRAIN_CONFIG['early_stopping'],
    restore_best_weights=SUN_TRAIN_CONFIG['restore_best_weights'],
    grad_clip_norm=SUN_TRAIN_CONFIG['grad_clip_norm'],
    stream_monitoring=SUN_TRAIN_CONFIG['stream_monitoring'],
    monitor=SUN_TRAIN_CONFIG['monitor'],
    modality_dropout=SUN_TRAIN_CONFIG['modality_dropout'],
    modality_dropout_start=SUN_TRAIN_CONFIG['modality_dropout_start'],
    modality_dropout_ramp=SUN_TRAIN_CONFIG['modality_dropout_ramp'],
    modality_dropout_rate=SUN_TRAIN_CONFIG['modality_dropout_rate'],
    gradient_monitoring=SUN_TRAIN_CONFIG['gradient_monitoring'],
    gradient_log_freq=SUN_TRAIN_CONFIG['gradient_log_freq'],
    track_integration_weights=SUN_TRAIN_CONFIG['track_integration_weights'],
    integration_snapshot_path=SUN_TRAIN_CONFIG['integration_snapshot_path'],
    integration_snapshot_freq=SUN_TRAIN_CONFIG['integration_snapshot_freq'],
)

# Merge warmup + full history if applicable
history = merge_histories(warmup_history, sun_history)

print("\n" + "=" * 60)
print("SUN FINE-TUNING COMPLETE!")
print("=" * 60)

## 21. Single-Stream Robustness Evaluation

In [ ]:
print("\n" + "=" * 60)
print("SINGLE-STREAM ROBUSTNESS EVALUATION (TEST SET)")
print("=" * 60)
print("\nTesting model performance with missing streams...\n")

# Evaluate with all streams (normal)
print("[1/3] Evaluating with BOTH streams (normal):")
results_both = model.evaluate(sun_test_loader, stream_monitoring=True)
print(f"      Accuracy: {results_both['accuracy']*100:.2f}%  MCA: {results_both['mean_class_accuracy']*100:.2f}%")

# Evaluate with RGB only (Depth blanked)
print("\n[2/3] Evaluating with RGB ONLY (Depth blanked):")
results_rgb_only = model.evaluate(sun_test_loader, stream_monitoring=True, blanked_streams={1})
print(f"      Accuracy: {results_rgb_only['accuracy']*100:.2f}%  MCA: {results_rgb_only['mean_class_accuracy']*100:.2f}%")

# Evaluate with Depth only (RGB blanked)
print("\n[3/3] Evaluating with DEPTH ONLY (RGB blanked):")
results_depth_only = model.evaluate(sun_test_loader, stream_monitoring=True, blanked_streams={0})
print(f"      Accuracy: {results_depth_only['accuracy']*100:.2f}%  MCA: {results_depth_only['mean_class_accuracy']*100:.2f}%")

print("\n" + "=" * 60)
print("ROBUSTNESS SUMMARY")
print("=" * 60)
print(f"\n  Both streams:  Acc={results_both['accuracy']*100:.2f}%  MCA={results_both['mean_class_accuracy']*100:.2f}%")
print(f"  RGB only:      Acc={results_rgb_only['accuracy']*100:.2f}%  MCA={results_rgb_only['mean_class_accuracy']*100:.2f}% (Depth missing)")
print(f"  Depth only:    Acc={results_depth_only['accuracy']*100:.2f}%  MCA={results_depth_only['mean_class_accuracy']*100:.2f}% (RGB missing)")

rgb_degradation = (results_both['accuracy'] - results_rgb_only['accuracy']) * 100
depth_degradation = (results_both['accuracy'] - results_depth_only['accuracy']) * 100

print(f"\n  Degradation when Depth missing: {rgb_degradation:+.2f}%")
print(f"  Degradation when RGB missing:   {depth_degradation:+.2f}%")
print("\n" + "=" * 60)

## 22. Test Set Evaluation + Pathway Analysis

In [ ]:
print("=" * 60)
print("TEST SET EVALUATION")
print("=" * 60)

# Evaluate on test set
results = model.evaluate(data_loader=sun_test_loader, stream_monitoring=True)

print(f"\nTest Results:")
print(f"  Loss: {results['loss']:.4f}")
print(f"  Overall Accuracy: {results['accuracy']*100:.2f}%")
print(f"  Mean Class Accuracy: {results['mean_class_accuracy']*100:.2f}%")

print(f"\nStream-Specific Performance:")
for i in range(len(SUN_MODEL_CONFIG['stream_input_channels'])):
    other = (i + 1) % 2
    solo_acc = results[f'stream_{other}_blanked_acc']
    print(f"  Stream{i} ({STREAM_LABELS[i]}) Solo Accuracy: {solo_acc*100:.2f}%")
    print(f"  Stream{i} ({STREAM_LABELS[i]}) Contribution: {results[f'stream_{i}_contribution']*100:+.2f}%")

# Pathway analysis
print(f"\n{'='*60}")
print("PATHWAY ANALYSIS")
print(f"{'='*60}")
print(f"\nAnalyzing stream pathways and integrated pathway contributions...")

pathway_analysis = model.analyze_pathways(data_loader=sun_test_loader)

print(f"\nSamples analyzed: {pathway_analysis['samples_analyzed']}")

# Accuracy
print("\nAccuracy:")
print(f"  Full model:      {pathway_analysis['accuracy']['full_model']*100:.2f}%")
for i in range(len(SUN_MODEL_CONFIG['stream_input_channels'])):
    acc = pathway_analysis['accuracy'][f'stream{i}_only']
    contrib = pathway_analysis['accuracy'][f'stream{i}_contribution']
    print(f"  {STREAM_LABELS[i]} only:       {acc*100:.2f}%  (contribution ratio: {contrib:.3f})")

# Loss
print("\nLoss:")
print(f"  Full model:      {pathway_analysis['loss']['full_model']:.4f}")
for i in range(len(SUN_MODEL_CONFIG['stream_input_channels'])):
    loss_i = pathway_analysis['loss'][f'stream{i}_only']
    loss_contrib = pathway_analysis['loss'][f'stream{i}_contribution']
    print(f"  {STREAM_LABELS[i]} only:       {loss_i:.4f}  (loss ratio: {loss_contrib:.3f})")

# Feature norms
print("\nFeature Norms (mean +/- std):")
for i in range(len(SUN_MODEL_CONFIG['stream_input_channels'])):
    mean = pathway_analysis['feature_norms'][f'stream{i}_mean']
    std = pathway_analysis['feature_norms'][f'stream{i}_std']
    print(f"  {STREAM_LABELS[i]}:        {mean:.4f} +/- {std:.4f}")
int_mean = pathway_analysis['feature_norms']['integrated_mean']
int_std = pathway_analysis['feature_norms']['integrated_std']
print(f"  Integrated:  {int_mean:.4f} +/- {int_std:.4f}")

# Training summary
print(f"\n{'='*60}")
print("TRAINING SUMMARY")
print(f"{'='*60}")
print(f"  Initial train loss: {history['train_loss'][0]:.4f}")
print(f"  Final train loss:   {history['train_loss'][-1]:.4f}")
print(f"  Initial train acc:  {history['train_accuracy'][0]*100:.2f}%")
print(f"  Final train acc:    {history['train_accuracy'][-1]*100:.2f}%")
print(f"  Test accuracy:      {results['accuracy']*100:.2f}%")
print(f"  Test MCA:           {results['mean_class_accuracy']*100:.2f}%")
print(f"  Total epochs:       {len(history['train_loss'])}")

print("\n" + "=" * 60)

## 23. Training Curves

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

# --- Row 1: Standard training curves (restored from original + adapted for no val set) ---

# Loss curve
axes[0, 0].plot(history['train_loss'], label='Train Loss', linewidth=2)
axes[0, 0].set_xlabel('Epoch', fontsize=12)
axes[0, 0].set_ylabel('Loss', fontsize=12)
axes[0, 0].set_title('Training Loss', fontsize=14, fontweight='bold')
axes[0, 0].legend(fontsize=11)
axes[0, 0].grid(True, alpha=0.3)

# Accuracy curve with per-stream curves
axes[0, 1].plot([acc*100 for acc in history['train_accuracy']], label='Full Model Train', linewidth=2, color='green')
if 'train_mca' in history and history['train_mca']:
    axes[0, 1].plot([m*100 for m in history['train_mca']], label='Train MCA', linewidth=2, color='darkorange', linestyle=':')

# Add per-stream curves (always available with stream_monitoring=True)
stream_train_colors = ['skyblue', 'lightcoral', 'gold', 'lightgreen', 'plum']
stream_val_colors = ['blue', 'red', 'orange', 'green', 'purple']
for i in range(len(SUN_MODEL_CONFIG['stream_input_channels'])):
    color_idx = i % len(stream_train_colors)
    axes[0, 1].plot([acc*100 for acc in history[f'stream_{i}_train_acc']],
                label=f'{STREAM_LABELS[i]} Train', linewidth=1, alpha=0.6, linestyle='--',
                color=stream_train_colors[color_idx])

axes[0, 1].set_xlabel('Epoch', fontsize=12)
axes[0, 1].set_ylabel('Accuracy (%)', fontsize=12)
axes[0, 1].set_yticks([20, 40, 60, 80, 100])
axes[0, 1].set_title('Training Accuracy\n(Full Model = Integrated Stream)', fontsize=14, fontweight='bold')
axes[0, 1].legend(fontsize=9, loc='lower right')
# Draw gridlines manually: alpha=0.3 for multiples of 10, alpha=0.2 for 5, 15, 25...
for y in range(0, 101, 10):
    axes[0, 1].axhline(y=y, color='gray', alpha=0.3, linewidth=0.5)
for y in range(5, 100, 10):
    axes[0, 1].axhline(y=y, color='gray', alpha=0.2, linewidth=0.5)
axes[0, 1].grid(True, axis='x', alpha=0.3)

# Learning rate curve with per-stream LRs
sampled_lrs = history['learning_rates'][::max(1, len(history['learning_rates'])//100)]
axes[0, 2].plot(sampled_lrs, linewidth=2, color='green', label='Base LR')

# Add per-stream LRs (always available with stream_monitoring=True)
lr_colors = ['blue', 'red', 'orange', 'purple', 'brown']
for i in range(len(SUN_MODEL_CONFIG['stream_input_channels'])):
    color_idx = i % len(lr_colors)
    axes[0, 2].plot(history[f'stream_{i}_lr'], linewidth=1, alpha=0.7, linestyle='--',
                color=lr_colors[color_idx], label=f'{STREAM_LABELS[i]} LR')

axes[0, 2].set_xlabel('Epoch', fontsize=12)
axes[0, 2].set_ylabel('Learning Rate', fontsize=12)
axes[0, 2].set_title('Learning Rate Schedule', fontsize=14, fontweight='bold')
axes[0, 2].legend(fontsize=9, loc='upper right')
axes[0, 2].grid(True, alpha=0.3)

# --- Row 2: New diagnostics ---

# Gradient norms over epochs (values are dicts with mean/max/min)
if 'gradient_norms' in history and history['gradient_norms']:
    grad_epochs = range(len(history['gradient_norms']))
    for i in range(len(SUN_MODEL_CONFIG['stream_input_channels'])):
        key = f'stream_{i}'
        norms = [d.get(key, {}).get('mean', 0) for d in history['gradient_norms']]
        color = stream_val_colors[i % len(stream_val_colors)]
        axes[1, 0].plot(grad_epochs, norms, label=f'{STREAM_LABELS[i]}', color=color, linewidth=1.5)
    shared_norms = [d.get('shared', {}).get('mean', 0) for d in history['gradient_norms']]
    axes[1, 0].plot(grad_epochs, shared_norms, label='Shared', color='gray', linewidth=1.5, linestyle='--')
    axes[1, 0].set_yscale('log')
    axes[1, 0].set_xlabel('Epoch', fontsize=12)
    axes[1, 0].set_ylabel('Gradient Norm (pre-clip, log)', fontsize=12)
    axes[1, 0].set_title('Per-Stream Gradient Norms (mean)', fontsize=14, fontweight='bold')
    axes[1, 0].legend(fontsize=9)
    axes[1, 0].grid(True, alpha=0.3)
else:
    axes[1, 0].text(0.5, 0.5, 'No gradient data\n(gradient_monitoring=False)', ha='center', va='center',
                    transform=axes[1, 0].transAxes, fontsize=12)
    axes[1, 0].set_title('Per-Stream Gradient Norms', fontsize=14, fontweight='bold')

contrib_keys = [f'stream_{i}_train_acc' for i in range(len(SUN_MODEL_CONFIG['stream_input_channels']))]
if contrib_keys[0] in history:
    n_streams = len(SUN_MODEL_CONFIG['stream_input_channels'])
    baseline_vals = history['train_accuracy']
    for i in range(n_streams):
        color = stream_val_colors[i % len(stream_val_colors)]
        other = (i + 1) % n_streams if n_streams == 2 else i
        other_vals = history[f'stream_{other}_train_acc']
        contrib = []
        epochs_eval = []
        for e, (other_acc, base) in enumerate(zip(other_vals, baseline_vals)):
            if not math.isnan(other_acc):
                contrib.append((base - other_acc) * 100)
                epochs_eval.append(e)
        axes[1, 1].plot(epochs_eval, contrib,
                       label=f'{STREAM_LABELS[i]}', color=color, linewidth=1.5, marker='o', markersize=3)
    axes[1, 1].axhline(y=0, color='gray', linestyle='--', alpha=0.5)
    axes[1, 1].set_xlabel('Epoch', fontsize=12)
    axes[1, 1].set_ylabel('Contribution (pp)', fontsize=12)
    axes[1, 1].set_title('Per-Stream Contribution\n(Baseline \u2212 Acc w/o Stream)', fontsize=14, fontweight='bold')
    axes[1, 1].legend(fontsize=9)
    axes[1, 1].grid(True, alpha=0.3)
else:
    axes[1, 1].text(0.5, 0.5, 'No stream data\n(stream_monitoring=False)', ha='center', va='center',
                    transform=axes[1, 1].transAxes, fontsize=12)
    axes[1, 1].set_title('Per-Stream Contribution', fontsize=14, fontweight='bold')




# Gradient health status summary
if 'gradient_health' in history and history['gradient_health']:
    axes[1, 2].axis('off')
    health_text = "Gradient Health Summary:\n\n"
    status_counts = {}
    for h in history['gradient_health']:
        status = h.get('status', 'unknown') if isinstance(h, dict) else str(h)
        status_counts[status] = status_counts.get(status, 0) + 1
    for status, count in sorted(status_counts.items(), key=lambda x: -x[1]):
        health_text += f"  {status}: {count} epochs\n"
    axes[1, 2].text(0.1, 0.9, health_text, transform=axes[1, 2].transAxes,
                    fontsize=10, verticalalignment='top', fontfamily='monospace')
    axes[1, 2].set_title('Gradient Health', fontsize=14, fontweight='bold')
else:
    axes[1, 2].text(0.5, 0.5, 'No gradient health data\n(gradient_monitoring=False)', ha='center', va='center',
                    transform=axes[1, 2].transAxes, fontsize=12)
    axes[1, 2].set_title('Gradient Health', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig(f"{sun_checkpoint_dir}/training_diagnostics.pdf", dpi=150, bbox_inches='tight')
plt.show()

print(f"Training diagnostics saved to: {sun_checkpoint_dir}/training_diagnostics.pdf")

## 24. Integration Weight Evolution

In [ ]:
evo_viz = IntegrationWeightEvolutionVisualizer(stream_labels=STREAM_LABELS)

# Stream backbone weight norm evolution
if 'stream_weight_norms' in history:
    evo_viz.plot_stream_weight_norms(history, save_path=f"{sun_checkpoint_dir}/stream_weight_evolution.pdf")
    print("Stream weight norm evolution saved.")
else:
    print("No stream weight norm data found.")

# Integration weight norm evolution
if 'integration_weight_norms' in history:
    evo_viz.plot_norm_evolution(history, save_path=f"{sun_checkpoint_dir}/integration_weight_evolution.pdf")
    print("Integration weight norm evolution saved.")
else:
    print("No integration weight norm data found.")

# Full weight snapshots
snapshot_dir = SUN_TRAIN_CONFIG.get('integration_snapshot_path')
if snapshot_dir and os.path.isdir(snapshot_dir) and os.listdir(snapshot_dir):
    evo_viz.plot_snapshot_heatmaps(snapshot_dir, save_path=f"{sun_checkpoint_dir}/integration_weight_snapshots.png")
    print("Integration weight snapshot heatmaps saved (full, PNG).")
    for layer_name in ['conv1', 'layer1']:
        evo_viz.plot_snapshot_heatmaps(
            snapshot_dir, layer_filter=layer_name,
            save_path=f"{sun_checkpoint_dir}/integration_weight_snapshots_{layer_name}.pdf"
        )
    print("Integration weight snapshot heatmaps saved (conv1 + layer1, PDF).")
else:
    print("No integration weight snapshots found.")

## 25. Save Results & Model

In [ ]:
print("=" * 60)
print("SAVING RESULTS")
print("=" * 60)

# Save training history as JSON
history_path = f"{sun_checkpoint_dir}/training_history.json"
with open(history_path, 'w') as f:
    # Build pathway analysis dict with all returned data
    pa_json = {
        'accuracy': {k: float(v) for k, v in pathway_analysis['accuracy'].items()},
        'loss': {k: float(v) for k, v in pathway_analysis['loss'].items()},
        'feature_norms': {k: float(v) for k, v in pathway_analysis['feature_norms'].items()},
        'samples_analyzed': pathway_analysis['samples_analyzed'],
    }

    json_history = {
        'train_loss': [float(x) for x in history['train_loss']],
        'train_accuracy': [float(x) for x in history['train_accuracy']],
        'learning_rates': [float(x) for x in history['learning_rates']],
        'model_config': SUN_MODEL_CONFIG,
        'dataset_config': {k: str(v) if not isinstance(v, (int, float, bool, type(None))) else v
                           for k, v in SUN_DATASET_CONFIG.items()},
        'optimizer_config': SUN_OPTIMIZER_CONFIG,
        'scheduler_config': SUN_SCHEDULER_CONFIG,
        'training_config': {k: str(v) if not isinstance(v, (int, float, bool, type(None))) else v
                            for k, v in SUN_TRAIN_CONFIG.items()},
        'augmentation_config': SUN_AUGMENTATION_CONFIG.to_dict(),
        'train_mca': [float(x) for x in history.get('train_mca', [])],
        'test_results': {
            'loss': float(results['loss']),
            'accuracy': float(results['accuracy']),
            'mean_class_accuracy': float(results.get('mean_class_accuracy', 0)),
        },
        'pathway_analysis': pa_json,
    }

    json.dump(json_history, f, indent=2)

print(f"Training history saved: {history_path}")

# Save final model
final_model_path = f"{sun_checkpoint_dir}/final_model.pt"
torch.save({
    'model_state_dict': model.state_dict(),
    'optimizer_state_dict': model.optimizer.state_dict(),
    'scheduler_state_dict': model.scheduler.state_dict() if model.scheduler else None,
    'config': SUN_MODEL_CONFIG,
    'history': history,
    'test_accuracy': results['accuracy']
}, final_model_path)

print(f"Final model saved: {final_model_path}")

# List saved files
print(f"\nAll results saved to: {sun_checkpoint_dir}")
!ls -lh {sun_checkpoint_dir}

print("\n" + "=" * 60)

## 26. Internal CNN Visualization Suite

Everything below runs on the **trained model** with the **test set**. Each cell is independent.

In [ ]:
# Aliases for visualization cells (same interface as standalone SUN notebook)
test_loader = sun_test_loader
train_loader = sun_train_loader
checkpoint_dir = sun_checkpoint_dir
class_names = sun_class_names
MODEL_CONFIG = SUN_MODEL_CONFIG

# --- 17a. Feature Map Visualization ---
# "What does the CNN see at each layer?"
# Three modes: full model, single-stream isolated, ablation
# Compare layer1 (early/texture) vs layer4 (late/semantic)

fm_viz = FeatureMapVisualizer(model, stream_labels=STREAM_LABELS)

# Get a single test sample
test_iter = iter(test_loader)
sample_batch = next(test_iter)
*stream_batches, labels = sample_batch
# Take first sample
stream_inputs = [s[0:1].to(model.device) for s in stream_batches]

print(f"Sample label: {labels[0].item()} ({class_names[labels[0].item()] if 'class_names' in dir() else '?'})")

for layer in ['layer1', 'layer4']:
    print(f"\n{'='*60}")
    print(f"  {layer.upper()} FEATURE MAPS")
    print(f"{'='*60}")

    # Mode 1: Full model view (all streams + integrated)
    print(f"\n--- Full Model View ({layer}) ---")
    fm_viz.visualize(stream_inputs, layer=layer, top_k=8,
                     save_path=f"{checkpoint_dir}/featuremaps_full_{layer}.png")

    # Mode 2: Per-stream isolated views
    for i, label in STREAM_LABELS.items():
        print(f"\n--- {label} Stream Isolated View ({layer}) ---")
        fm_viz.visualize(stream_inputs, layer=layer, mode='stream', stream_idx=i, top_k=8,
                         save_path=f"{checkpoint_dir}/featuremaps_{label.lower()}_{layer}.png")

    # Mode 3: Ablation (what happens when we remove a stream?)
    for i, label in STREAM_LABELS.items():
        print(f"\n--- Ablation: {label} Blanked ({layer}) ---")
        fm_viz.visualize(stream_inputs, layer=layer, mode='ablation', stream_idx=i, top_k=8,
                         save_path=f"{checkpoint_dir}/featuremaps_ablation_{label.lower()}_{layer}.png")

# Batch-averaged feature maps at both layers
for layer in ['layer1', 'layer4']:
    print(f"\n--- Batch-Averaged Feature Maps ({layer}, 32 samples) ---")
    fm_viz.visualize_batch(test_loader, layer=layer, n=32, top_k=8,
                           save_path=f"{checkpoint_dir}/featuremaps_batch_avg_{layer}.png")

print("\nFeature map visualizations complete!")

In [ ]:
# --- 17b. Stream Contribution Decomposition ---
# THE unique LINet3 visualization: how much does each stream contribute
# to each neuron's activation in the integrated pathway?

contrib_viz = StreamContributionVisualizer(model, stream_labels=STREAM_LABELS)

# Single image contribution at layer4
print("--- Stream Contributions (layer4, single sample) ---")
contrib_viz.visualize(stream_inputs, layer='layer4',
                      save_path=f"{checkpoint_dir}/contributions_layer4.pdf")

# Batch-averaged contributions (more representative)
print("\n--- Batch-Averaged Contributions (layer4, 32 samples) ---")
contrib_viz.visualize_batch(test_loader, layer='layer4', n=32,
                            save_path=f"{checkpoint_dir}/contributions_batch_layer4.pdf")

# Multi-layer comparison
for layer in ['layer1', 'layer2', 'layer3', 'layer4']:
    print(f"\n--- Contributions at {layer} ---")
    contrib_viz.visualize(stream_inputs, layer=layer,
                          save_path=f"{checkpoint_dir}/contributions_{layer}.pdf")

print("\nStream contribution decomposition complete!")

In [ ]:
# --- 17c. Stream-Decomposed Grad-CAM ---
# Where does each stream focus its attention?

gradcam = StreamGradCAM(model, stream_labels=STREAM_LABELS)

# Integrated Grad-CAM (standard: where does the full model look?)
print("--- Integrated Grad-CAM (layer4) ---")
gradcam.visualize(stream_inputs, layer='layer4', mode='integrated',
                  save_path=f"{checkpoint_dir}/gradcam_integrated_layer4.png")

# Per-stream isolated Grad-CAM (where does each stream look independently?)
for i, label in STREAM_LABELS.items():
    print(f"\n--- {label} Stream Grad-CAM (layer4) ---")
    gradcam.visualize(stream_inputs, layer='layer4', mode='stream', stream_idx=i,
                      save_path=f"{checkpoint_dir}/gradcam_{label.lower()}_layer4.png")

# Decomposed mode: contribution maps weighted by Grad-CAM importance
print("\n--- Decomposed Grad-CAM (layer4) ---")
gradcam.visualize(stream_inputs, layer='layer4', mode='decomposed',
                  save_path=f"{checkpoint_dir}/gradcam_decomposed_layer4.png")

# Multi-layer Grad-CAM (early=texture, late=semantics)
for layer in ['layer2', 'layer3', 'layer4']:
    print(f"\n--- Integrated Grad-CAM at {layer} ---")
    gradcam.visualize(stream_inputs, layer=layer, mode='integrated',
                      save_path=f"{checkpoint_dir}/gradcam_integrated_{layer}.png")

print("\nGrad-CAM visualizations complete!")

In [ ]:
# --- 17d. Integration Weight Visualization ---
# Visualize the learned integration_from_streams weights per layer

iw_viz = IntegrationWeightVisualizer(model, stream_labels=STREAM_LABELS)

# Weight heatmaps per layer and stream
print('--- Integration Weights (Heatmaps) ---')
iw_viz.visualize_weights(save_path=f'{checkpoint_dir}/integration_weights.png')

# Cross-stream comparison (relative weight magnitudes per layer)
print('\n--- Cross-Stream Weight Comparison ---')
iw_viz.visualize_cross_stream(save_path=f'{checkpoint_dir}/integration_cross_stream.pdf')

# Effective rank via SVD (how low-dimensional is the integration?)
print('\n--- Effective Rank (SVD) ---')
ranks = iw_viz.compute_effective_rank()
for layer, r in ranks.items():
    print(f'  {layer}: {[f"{x:.1f}" for x in r]}')

print('\nIntegration weight visualization complete!')

In [ ]:
# --- 17e. Stream Redundancy Analysis ---
# Are RGB and Depth learning the same features? Or complementary ones?
# Uses centered cosine similarity between stream feature maps at each layer.

redundancy = StreamRedundancyAnalyzer(model, stream_labels=STREAM_LABELS)

print('--- Stream Redundancy (Centered Cosine Similarity) ---')
sim_results = redundancy.analyze(
    test_loader,
    n=128,  # Average over 128 samples
    save_path=f'{checkpoint_dir}/stream_redundancy.pdf'
)

# Print similarity matrices
for layer_name, sim_matrix in sim_results.items():
    print(f'\n{layer_name}:')
    for i in range(sim_matrix.shape[0]):
        row = '  '.join(f'{sim_matrix[i,j]:.3f}' for j in range(sim_matrix.shape[1]))
        print(f'  {STREAM_LABELS.get(i, f"S{i}")}: {row}')

print('\nStream redundancy analysis complete!')

In [ ]:
# --- 17f. Per-Class Stream Dominance ---
# Which scenes rely on RGB vs Depth?
# "Depth matters more for bathrooms, RGB dominates corridors"

# Build class name mapping
class_name_map = {i: name for i, name in enumerate(class_names)} if 'class_names' in dir() else None

dominance = PerClassDominanceAnalyzer(model, stream_labels=STREAM_LABELS)

print('--- Per-Class Stream Dominance (layer4) ---')
class_dominance = dominance.analyze(
    test_loader,
    layer='layer4',
    class_names=class_name_map,
    save_path=f'{checkpoint_dir}/per_class_dominance.pdf'
)

# Print per-class ratios
print('\nPer-class stream contribution ratios:')
for cls_idx, ratios in sorted(class_dominance.items()):
    name = class_name_map[cls_idx] if class_name_map else f'Class {cls_idx}'
    ratio_str = ', '.join(f'{STREAM_LABELS.get(i, f"S{i}")}: {r:.2%}' for i, r in enumerate(ratios))
    print(f'  {name}: {ratio_str}')

print('\nPer-class dominance analysis complete!')

In [ ]:
# --- 17g. Misclassification Analysis + Sample Comparison ---
# Find misclassified samples and compare with correctly classified ones

print('--- Finding Misclassified Samples ---')
misclassified = find_misclassified(model, test_loader, n=10)

print(f'Found {len(misclassified)} misclassified samples:')
for i, mc in enumerate(misclassified[:5]):
    true_name = class_names[mc['true_label']] if 'class_names' in dir() else str(mc['true_label'])
    pred_name = class_names[mc['predicted_label']] if 'class_names' in dir() else str(mc['predicted_label'])
    print(f'  [{i}] True: {true_name}, Predicted: {pred_name}, Confidence: {mc["confidence"]:.2%}')

# Grad-CAM on first misclassified sample
if misclassified:
    mc_sample = misclassified[0]
    mc_inputs = [s.to(model.device) for s in mc_sample['stream_inputs']]
    true_name = class_names[mc_sample['true_label']] if 'class_names' in dir() else str(mc_sample['true_label'])
    pred_name = class_names[mc_sample['predicted_label']] if 'class_names' in dir() else str(mc_sample['predicted_label'])
    print(f'\n--- Grad-CAM on Misclassified: True={true_name}, Pred={pred_name} ---')
    gradcam.visualize(mc_inputs, layer='layer4', mode='decomposed',
                      save_path=f'{checkpoint_dir}/gradcam_misclassified_0.png')

# Compare correct vs misclassified from same class
if misclassified:
    target_class = misclassified[0]['true_label']
    print(f'\n--- Finding correctly classified sample from class {class_names[target_class] if "class_names" in dir() else target_class} ---')

    # Find a correctly classified sample from the same class
    correct_sample = None
    model.eval()
    with torch.no_grad():
        for batch_data in test_loader:
            *stream_batches, targets = batch_data
            stream_batches_dev = [s.to(model.device) for s in stream_batches]
            targets_dev = targets.to(model.device)
            logits = model(stream_batches_dev)
            preds = logits.argmax(dim=1)
            # Find correctly classified samples of the target class
            mask = (targets_dev == target_class) & (preds == target_class)
            if mask.any():
                idx = mask.nonzero(as_tuple=True)[0][0].item()
                correct_sample = {
                    'stream_inputs': [s[idx:idx+1].cpu() for s in stream_batches],
                    'true_label': target_class,
                    'predicted_label': target_class,
                    'confidence': torch.softmax(logits[idx], dim=0)[target_class].item(),
                }
                break

    if correct_sample is not None:
        print(f'  Found correct sample (confidence: {correct_sample["confidence"]:.2%})')
        print('\n--- Correct vs Misclassified Comparison ---')
        compare_samples(
            model,
            correct_sample=correct_sample,
            misclassified_sample=misclassified[0],
            layer='layer4',
            stream_labels=STREAM_LABELS,
            save_path=f'{checkpoint_dir}/compare_samples.png'
        )
    else:
        print('  No correctly classified sample found for this class.')

print('\nMisclassification analysis complete!')

In [ ]:
# --- 17h. Train vs Test Activation Divergence ---
# Does the model see different activation distributions on train vs test?
# Uses MMD (Maximum Mean Discrepancy) per layer.

div_analyzer = ActivationDivergenceAnalyzer(model)

print('--- Train vs Test Activation Divergence ---')
divergence = div_analyzer.analyze(
    train_loader,
    test_loader,
    n=128,
    save_path=f'{checkpoint_dir}/activation_divergence.pdf'
)

for layer_name, metrics in divergence.items():
    print(f'  {layer_name}: MMD={metrics["mmd"]:.4f}')

print('\nActivation divergence analysis complete!')

In [ ]:
# --- 17i. BN Stats Reset Experiment (Oracle Diagnostic) ---
# WARNING: This is a DIAGNOSTIC tool, not a deployable fix.
# It recomputes BN running stats on test data (oracle) to check if
# BN statistics drift causes the generalization gap.

# Save original accuracy
original_test_results = model.evaluate(test_loader)
original_acc = original_test_results['accuracy']
print(f'Original test accuracy: {original_acc*100:.2f}%')

# Control: recompute BN stats on TRAIN set (should be ~same)
print('\n--- Control: Recompute BN stats on TRAIN set ---')
model_control = copy.deepcopy(model)
reset_bn_stats(model_control, train_loader)
control_results = model_control.evaluate(test_loader)
control_acc = control_results['accuracy']
print(f'After train BN reset: {control_acc*100:.2f}% (delta: {(control_acc-original_acc)*100:+.2f}%)')

# Oracle: recompute BN stats on TEST set
print('\n--- Oracle: Recompute BN stats on TEST set ---')
model_oracle = copy.deepcopy(model)
reset_bn_stats(model_oracle, test_loader)
oracle_results = model_oracle.evaluate(test_loader)
oracle_acc = oracle_results['accuracy']
print(f'After test BN reset (oracle): {oracle_acc*100:.2f}% (delta: {(oracle_acc-original_acc)*100:+.2f}%)')

# Interpretation
print('\n--- Interpretation ---')
oracle_delta = (oracle_acc - original_acc) * 100
if abs(oracle_delta) > 2:
    print(f'BN stats drift accounts for ~{oracle_delta:+.1f}% of the gap.')
    print('Consider: test-time BN adaptation, larger batch size, or more training data.')
else:
    print(f'BN stats drift is minimal ({oracle_delta:+.1f}%). Gap is likely from other sources.')

del model_control, model_oracle  # Free memory
print('\nBN reset experiment complete!')

## 27. Summary

**Phase 1 (ScanNet Pretrain):** All outputs saved to ScanNet checkpoint directory on Drive.

**Phase 2 (SUN Fine-tune):** All outputs saved to SUN checkpoint directory on Drive.

**Saved models:** `best_model.pt`, `final_model.pt` (for each phase)

**Saved data:** `training_history.json`, `integration_snapshots/` (for each phase)

**Saved visualizations:** Training diagnostics, integration weight evolution, feature maps, Grad-CAM, stream contributions, redundancy analysis, per-class dominance, misclassification analysis, activation divergence, BN stats reset.